# 05b · LOIO training — UPerNet (ConvNeXt-T) (30 runs)
Specification: `specifications/01_spec_loio_calibration_traits.md`. Runs in parallel with notebook 05a (the other architecture).

**Keep the same GPU type for this notebook for all its runs** (also after a reconnect), so that the three losses of every fold are trained on identical hardware. If Colab disconnects, just *Run all* again: finished runs are skipped, an interrupted run resumes.

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
from gwfss_stem import plan, engine
import pandas as pd

FILTER = dict(backbone='upernet_convnext_t')

runs = plan.select(plan.loio_plan(), **FILTER)
def status(r):
    d = os.path.join(RESULTS, engine.run_name({**engine.DEFAULTS, **r}))
    if os.path.exists(f'{d}/final.pt'): return 'done'
    if os.path.exists(f'{d}/RUNNING'): return 'running/locked'
    if os.path.exists(f'{d}/last.pt'): return 'partial'
    return 'pending'
table = pd.DataFrame([{**r, 'status': status(r)} for r in runs])
display(table.status.value_counts()); display(table)

In [ ]:
for r in runs:
    cfg = {k: v for k, v in r.items() if k != 'sq'}
    run_dir = engine.train_run(cfg, DATA_ROOT, SPLIT_FILE, RESULTS)
    if run_dir:
        engine.evaluate_run(run_dir, DATA_ROOT, SPLIT_FILE, subset='test', tag='ss')